# Week 1 - Foundations: Linear Systems & Eigen-Images for Recognition

Almost every method in this course reduces to two questions about a matrix:
*how do I solve* $A\mathbf{x} = \mathbf{b}$, and *what are the natural axes of my
data?* Week 1 builds both and fuses them into a classical recognition pipeline.
We solve a linear system two ways -- watching when the answer can be trusted --
then turn to the **eigen-image** idea: representing a whole library of images in
a compact basis of a few "prototype" pictures found by principal component
analysis. The famous version is *eigenfaces*; here, on **real**
peripheral-blood-cell crops (BloodMNIST, via the course data layer), we build
**eigen-cells** and use them to compress, reconstruct, and classify.

The course's discipline is already here: never trust a number you haven't
checked. Every step is quantified -- the conditioning of the solve, the variance
a few modes capture, reconstruction error versus basis size, and classifier
accuracy on data it never saw during fitting.

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation*, 2nd ed.,
Chapter 2 (linear systems, least squares, the SVD, and the eigenface
recognition pipeline). Read it for the derivations; the treatment below is in
our own terms and runs against our own fixtures and the BloodMNIST library.

**Learning goals.**

- Solve a small linear system $A\mathbf{x}=\mathbf{b}$ directly and iteratively,
  and use the condition number to judge how much to trust the solution.
- Build an eigen-image basis by mean-centering an image library and taking its
  principal components with the singular value decomposition (`numpy.linalg.svd`).
- Read explained-variance ratios to decide how many modes reach 90/95/99% of
  the variance, and relate that to reconstruction error.
- Classify images in the compact eigen-basis with a nearest-neighbour rule,
  measured on the official held-out test partition, and state confidence honestly.


```{admonition} Which paradigm?
:class: note
**Data-driven.** You never write down a model of what a blood cell should look like. You mean-center the BloodMNIST microscopy crops, let the SVD hand you the eigen-cells -- the axes along which the images actually vary -- then reconstruct and classify in that learned basis with a nearest-neighbour rule. Those coordinates come inductively from the pixels, not from cell biology or the microscope's optics. That puts this week at the data-driven extreme of the course; next week makes the opposite, mechanistic move, committing to a dose-response model and fitting its parameters.
```


## Setup

We seed every random number generator and apply the course plotting style, so
the figures and numbers below are identical from a cold kernel.

In [ ]:
# Colab setup: install the ddm4bio course library.
# No-op when ddm4bio is already importable (e.g. the course-site build), so
# this cell is safe everywhere. It is hidden from the rendered site via the
# "remove-cell" tag, but runs when this notebook is opened in Google Colab.
try:
    import ddm4bio  # noqa: F401
except ModuleNotFoundError:
    %pip install -q "ddm4bio @ git+https://github.com/symbiont-ai/ddm4bio.git"
    import ddm4bio  # noqa: F401

In [ ]:
import numpy as np

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
set_style()

print(f"ddm4bio version: {ddm4bio.__version__}")

## 1. Solving $A\mathbf{x}=\mathbf{b}$ two ways, and the role of conditioning

A linear system asks: which combination of the columns of $A$ reproduces the
observation $\mathbf{b}$? For a small, well-behaved square system there are two
broad strategies. A **direct** solver factors $A$ and then solves by
substitution; because our $A$ is symmetric positive-definite -- and we tell SciPy so with
`assume_a="pos"` -- it uses an SPD-specialized factorization (Cholesky) rather than general
Gaussian elimination / LU. Either way it returns the answer in a fixed number of arithmetic
steps. An **iterative** solver starts from a guess and refines it -- here we use
conjugate gradient, a standard choice for large sparse symmetric positive-definite
systems. On a small dense problem the two should agree
to machine-ish precision; the point of showing both is to make their equivalence
concrete before we ever rely on either.

We construct a symmetric positive-definite $A$ (conjugate gradient needs
symmetry and positive-definiteness) and a known true solution, so we can measure
each solver against ground truth rather than against itself.

In [ ]:
import inspect

from scipy.linalg import solve as direct_solve
from scipy.sparse.linalg import cg

rng = np.random.default_rng(0)

n = 6
# Build a well-conditioned symmetric positive-definite matrix A = M M^T + nI
# (the nI shift lifts the eigenvalues, keeping the condition number small).
M = rng.standard_normal((n, n))
A = M @ M.T + n * np.eye(n)

x_true = rng.standard_normal(n)     # the answer we want to recover
b = A @ x_true                      # the observation the solvers are given

x_direct = direct_solve(A, b, assume_a="pos")

# SciPy renamed the relative-tolerance keyword from `tol` to `rtol` in 1.12;
# pick whichever this install exposes so the lesson runs on either version.
tol_kw = "rtol" if "rtol" in inspect.signature(cg).parameters else "tol"
x_iter, info = cg(A, b, maxiter=1000, atol=1e-12, **{tol_kw: 1e-10})

print(f"Direct solver error   ||x_hat - x_true||: {np.linalg.norm(x_direct - x_true):.2e}")
print(f"Iterative (CG) error  ||x_hat - x_true||: {np.linalg.norm(x_iter - x_true):.2e}")
print(f"CG converged (info==0): {info == 0}")
print(f"Direct vs iterative agree: {np.allclose(x_direct, x_iter, atol=1e-6)}")

Both solvers recover the true $\mathbf{x}$ and agree with each other. That is
the *easy* case. The reason we can trust the answer is not the algorithm but the
**conditioning** of $A$: the condition number $\kappa(A)$ measures how much a
small perturbation in $\mathbf{b}$ (rounding, measurement noise) can be
amplified in the solution $\mathbf{x}$. A well-conditioned matrix has
$\kappa$ near 1; an ill-conditioned one can turn a tiny input error into a
catastrophic output error, and *no* solver -- direct or iterative -- can rescue
you.

In [ ]:
# Contrast our well-conditioned A with a deliberately near-singular matrix.
A_ill = np.vander(np.linspace(1.0, 2.0, n), n)   # Vandermonde: notoriously ill-conditioned

kappa_good = np.linalg.cond(A)
kappa_bad = np.linalg.cond(A_ill)

# Probe error amplification: perturb b slightly and see how much x moves.
b_pert = b + 1e-8 * rng.standard_normal(n)
x_pert = direct_solve(A, b_pert, assume_a="pos")
rel_out = np.linalg.norm(x_pert - x_direct) / np.linalg.norm(x_direct)
rel_in = np.linalg.norm(b_pert - b) / np.linalg.norm(b)

print(f"cond(A) well-conditioned : {kappa_good:8.1f}")
print(f"cond(A) near-singular    : {kappa_bad:8.2e}")
print(f"Input relative perturbation : {rel_in:.2e}")
print(f"Output relative change      : {rel_out:.2e}")
print(f"Observed amplification      : {rel_out / rel_in:6.1f}x  (bounded by kappa = {kappa_good:.1f})")

**QC note.** The observed amplification factor sits comfortably below
$\kappa(A)$ for the well-conditioned matrix, exactly as the theory bounds it. A
Vandermonde matrix of the same size has a condition number many orders of
magnitude larger -- solving a system built on it would mean surrendering most of
your significant digits to round-off. The lesson for the rest of the course:
before trusting any solve, check the condition number.

## 2. From pixels to an image library

We now switch from a single linear system to a *library* of images. Here the
library is **real**: we pull BloodMNIST -- peripheral-blood-cell microscopy
crops from MedMNIST v2 -- through the course data layer,
`get_dataset("bloodmnist")`. Each crop is a small RGB image carrying an integer
cell-type label. We convert every crop to grayscale (averaging the colour
channels), flatten it to a vector, and take a *seeded* few-hundred-image
subsample so the whole lesson runs briskly. (The **eigen-cells** themselves come next, once we apply PCA to this library.)
The loader returns real data when it can reach the source and a labelled bundled
fallback with the *same payload shape* otherwise, so the pipeline below runs
identically either way -- and we print which one we got.

In [ ]:
from ddm4bio.datasets import get_dataset

ds = get_dataset("bloodmnist", seed=0)
print(f"Data source : {ds.source}")
print(f"Provenance  : {ds.provenance}")

# BloodMNIST ships an OFFICIAL train / validation / test split (a 7:1:2 partition), so we
# use it rather than re-splitting: the training partition drives the eigen-basis and the
# classifier, and the official test partition is used only to score, never to fit. We take a
# seeded subsample of each so the lesson runs briskly, and average the colour axis to grayscale.
rng_sub = np.random.default_rng(0)


def take(images, labels, n):
    idx = rng_sub.choice(images.shape[0], size=min(n, images.shape[0]), replace=False)
    gray = images[idx].mean(axis=-1)                       # (n, H, W) grayscale
    return gray, gray.reshape(gray.shape[0], -1).astype(float), labels[idx].ravel()


images, X, y = take(ds.payload["train_images"], ds.payload["train_labels"], 400)
_, X_test, y_test = take(ds.payload["test_images"], ds.payload["test_labels"], 400)
img_h, img_w = images.shape[1], images.shape[2]
classes = np.unique(np.concatenate([y, y_test]))
n_classes = classes.size

print(f"Training library : {X.shape[0]} images of {img_h}x{img_w} pixels ({n_classes} cell classes)")
print(f"Official test set: {X_test.shape[0]} held-out images (used only to score)")
print(f"Flattened feature matrix X: {X.shape} (samples x pixels)")

# BloodMNIST publishes named cell types (MedMNIST v2; Acevedo et al., 2020). We label the
# real crops with these; the offline fallback is synthetic, so it keeps generic class ids.
BLOODMNIST_CLASSES = {
    0: "basophil", 1: "eosinophil", 2: "erythroblast", 3: "immature granulocyte",
    4: "lymphocyte", 5: "monocyte", 6: "neutrophil", 7: "platelet",
}


def class_label(i):
    return BLOODMNIST_CLASSES[int(i)] if ds.source == "real" else f"class {int(i)}"

A quick look at a handful of raw "cells" from the library. This is what the
recognizer has to work with -- low-resolution, variable, and noisy.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 6, figsize=(11, 3.4))
for ax, img, label in zip(axes.flat, images, y):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(class_label(label), fontsize=7)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f"Twelve crops from the {img_h}x{img_w} blood-cell library")
fig;

## 3. Building the eigen-image basis (the ground-truth-adjacent check)

The eigen-image recipe is exactly PCA on the image library:

1. **Mean-center** -- subtract the average image so the basis describes
   *deviations* from the mean, not the mean itself.
2. **Take principal components** -- the right singular vectors of the centered
   data matrix are the eigen-images: orthogonal full-resolution pixel patterns (one weight per
   image pixel), ordered by how much library variance each explains.
3. **Project** -- every image becomes a short vector of coordinates in this
   basis (its PCA scores).

Because PCA is a *deterministic* linear algebra operation (an SVD), its "ground
truth" is self-checking: the explained-variance ratios are guaranteed to be
non-negative and to sum to one, and the top modes must reconstruct the data
better than any other orthogonal basis of the same size. We verify those
invariants explicitly rather than take them on faith.

In [ ]:
# Mean image and centered library.
mean_image = X.mean(axis=0)
X_centered = X - mean_image

# The eigen-images ARE the principal components: the right singular vectors of the centered
# library. NumPy's SVD returns them directly, ordered by how much variance each explains.
_, singular_values, vt = np.linalg.svd(X_centered, full_matrices=False)
evr = singular_values**2 / np.sum(singular_values**2)   # explained-variance ratio per mode

print(f"Explained-variance ratios sum to 1: {np.isclose(evr.sum(), 1.0)}")
print(f"All ratios non-negative and non-increasing: "
      f"{np.all(evr >= 0) and np.all(np.diff(evr) <= 1e-12)}")
print(f"Variance captured by mode 1 alone : {evr[0]:.1%}")
print(f"Variance captured by top 10 modes : {evr[:10].sum():.1%}")

The scree curve shows how fast the explained variance decays. Unlike the sharp
rank-2 elbow of a purely synthetic fixture, real image data has a *gentle*
shoulder: a handful of modes dominate, but a long tail of small modes carries
the fine detail that distinguishes similar cell types.

In [ ]:
from ddm4bio.viz.plots import scree_plot

ax = scree_plot(evr[:20])       # first 20 modes; the tail is a slow decay to zero
ax.set_title("Scree plot: explained variance of the top 20 eigen-images")
ax.figure;

Now the eigen-images themselves. Each is a full-resolution pixel pattern reshaped back to the image grid. The first few look like smooth blobs that capture gross cell shape and brightness; later ones encode progressively finer, higher-frequency contrasts.
Any library image is a weighted sum of the mean image plus these patterns.

In [ ]:
from ddm4bio.viz.plots import mode_grid

# The top eigen-images are the leading rows of Vt from the SVD computed above.
eigen_images = vt[:8]           # top 8 eigen-images, each length H*W

fig = mode_grid(eigen_images, shape=(img_h, img_w), ncols=4)
fig.suptitle("Top 8 eigen-images ('eigen-cells')")
fig;

## 4. Reconstruction error vs. number of modes

How many eigen-images do we actually need? Project each image onto the top $k$
modes, reconstruct it, and measure the error. As $k$ grows the reconstruction
tightens; the useful question is where the curve flattens -- the point past
which extra modes buy little fidelity. We report the relative $L_2$
reconstruction error with a short `rel_l2` helper, and
overlay the variance-captured milestones (90/95/99%).

In [ ]:
def reconstruct_with_k(X_centered, vt, k):
    """Project onto the top-k eigen-images and map back to pixel space."""
    basis = vt[:k]                      # (k, n_pixels)
    scores = X_centered @ basis.T       # (n_samples, k)
    return scores @ basis               # (n_samples, n_pixels), centered reconstruction


def rel_l2(reference, approx):
    """Relative L2 (Frobenius) reconstruction error."""
    return float(np.linalg.norm(reference - approx) / np.linalg.norm(reference))


# Variance milestones first, so the mode sweep and its plot span the real basis.
cum_evr = np.cumsum(evr)


def modes_for(threshold):
    return int(np.searchsorted(cum_evr, threshold) + 1)


k90, k95, k99 = modes_for(0.90), modes_for(0.95), modes_for(0.99)
n_modes = vt.shape[0]

# A sweep that spans the whole basis -- adapting to the real 28x28 data (hundreds of modes)
# and the 8x8 offline fallback alike -- and always includes the variance milestones.
base = [1, 2, 4, 8, 16, 32, 64, 128, 256]
k_values = sorted({k for k in base if k < n_modes} | {k90, k95, k99, n_modes})
errors = [rel_l2(X_centered, reconstruct_with_k(X_centered, vt, k)) for k in k_values]

print(f"Modes to reach 90% variance: {k90}")
print(f"Modes to reach 95% variance: {k95}")
print(f"Modes to reach 99% variance: {k99}  (of {n_modes} total)")
for k, e in zip(k_values, errors):
    print(f"  k={k:3d}:  relative L2 reconstruction error = {e:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, errors, marker="o", linewidth=1.5)
ax.set_xscale("log")
for (k, name), y_lab in zip([(k90, "90%"), (k95, "95%"), (k99, "99%")], [0.62, 0.40, 0.62]):
    ax.axvline(k, color="0.6", linestyle="--", linewidth=1)
    ax.text(k, y_lab, f"{name}\n(k={k})", fontsize=8, color="0.35", ha="center",
            backgroundcolor="white")
ax.set_xlabel("Number of eigen-images (k, log scale)")
ax.set_ylabel("Relative L2 reconstruction error")
ax.set_title("Reconstruction error falls as the eigen-basis grows")
fig;

A visual confirmation: the same cell crop reconstructed from an increasing
number of modes. With only a few eigen-images it is a smudge; it takes on the
order of a hundred modes to sharpen on this real library, and past the 99%
variance milestone the extra modes change little.

In [ ]:
sample_idx = 0
ks_to_show = sorted({1, 8, k95, k99, n_modes})
fig, axes = plt.subplots(1, len(ks_to_show) + 1, figsize=(2.0 * (len(ks_to_show) + 1), 2.2))
axes[0].imshow(images[sample_idx], cmap="gray_r")
axes[0].set_title("original", fontsize=9)
axes[0].set_xticks([]); axes[0].set_yticks([])
for ax, k in zip(axes[1:], ks_to_show):
    recon = reconstruct_with_k(X_centered, vt, k)[sample_idx] + mean_image
    ax.imshow(recon.reshape(img_h, img_w), cmap="gray_r")
    ax.set_title(f"k={k}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f"Reconstructing one blood-cell crop ({class_label(y[sample_idx])}) from k eigen-images")
fig;

## 5. Recognition on the official held-out test set

The payoff: classification in the compact basis, scored honestly. BloodMNIST ships an
official train / test split, so we use it rather than re-splitting the training images: we
learn the eigen-basis, the mean image, and the mode count from the **training** partition
alone, then classify each image of the **official test** partition by its nearest training
neighbour. No preprocessing, basis, mode count, or classifier ever sees the test set.

A caveat worth stating plainly: this bounds *memorization*, not every optimistic bias. The
check below rules out pixel-identical crops crossing the split, but without donor or
acquisition identifiers we cannot rule out the same donor, near-duplicate crops, or shared
staining and imaging conditions spanning the two partitions -- so we call the estimate an
honest held-out one, not "leakage-free."

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Sanity check across the official boundary: no test crop is byte-identical to a training
# crop. This rules out pixel-identical duplicates only -- the interpretation block below
# states what it cannot rule out.
train_rows = {row.tobytes() for row in X}
n_dup = sum(row.tobytes() in train_rows for row in X_test)
print(f"Training library: {X.shape[0]} images   Official test set: {X_test.shape[0]} images")
print(f"Test crops byte-identical to a training crop: {n_dup}")

# The eigen-basis, mean, and mode count were all fit on the TRAINING partition (Sections
# 3-4); k is 95% of the training-library variance, chosen without ever touching the test set.
k_class = k95
Z_train = (X - mean_image) @ vt[:k_class].T
Z_test = (X_test - mean_image) @ vt[:k_class].T   # TRAIN mean & basis applied to the test set
print(f"Classifying in a {k_class}-dimensional eigen-basis (down from {X.shape[1]} raw pixels).")

In [ ]:
knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(Z_train, y)
acc_eigen = knn.score(Z_test, y_test)

# Baseline: the same 1-NN rule on raw pixels, for an honest comparison.
knn_raw = KNeighborsClassifier(n_neighbors=1)
knn_raw.fit(X, y)
acc_raw = knn_raw.score(X_test, y_test)

print(f"1-NN accuracy in the {k_class}-mode eigen-basis : {acc_eigen:.3f}")
print(f"1-NN accuracy on raw {X.shape[1]} pixels (baseline)   : {acc_raw:.3f}")
print(f"Dimensionality reduction: {X.shape[1]} -> {k_class} "
      f"({100 * k_class / X.shape[1]:.0f}% of the features), "
      f"accuracy change {acc_eigen - acc_raw:+.3f}")

The confusion matrix shows *where* the recognizer struggles -- typically among
cell types with similar morphology and staining. This is exactly the kind of
honest, class-resolved diagnostic that a single accuracy number hides.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, knn.predict(Z_test), labels=classes)
fig, ax = plt.subplots(figsize=(6.8, 5.8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xlabel("Predicted cell type")
ax.set_ylabel("True cell type")
ax.set_title(f"1-NN confusion matrix in the eigen-basis (k={k_class})")
ax.set_xticks(range(n_classes)); ax.set_yticks(range(n_classes))
ax.set_xticklabels([class_label(c) for c in classes], rotation=45, ha="right", fontsize=7)
ax.set_yticklabels([class_label(c) for c in classes], fontsize=7)
for i in range(n_classes):
    for j in range(n_classes):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    fontsize=7, color="0.2" if cm[i, j] < cm.max() / 2 else "white")
fig.colorbar(im, ax=ax, fraction=0.046, label="count")
fig;

**Why the held-out test set matters.** We fit the eigen-basis, the mean image, the number of
modes, and the classifier using only the training partition, then evaluated on the official
test partition, which none of that fitting had touched. The reported accuracy is therefore an
estimate of performance on *new* images, not a memorization score. Fitting the representation on the
full (pooled train-and-test) data can bias the test estimate, often optimistically, because
information from the test distribution enters the fitted pipeline -- though on any single
finite split a contaminated analysis can move either way.

In [ ]:
from ddm4bio.interpret import interpretation_block

acc_se = float(np.sqrt(acc_eigen * (1.0 - acc_eigen) / y_test.size))
acc_lo, acc_hi = acc_eigen - 1.96 * acc_se, acc_eigen + 1.96 * acc_se

block = interpretation_block(
    claim=(
        f"An eigen-image basis of {k_class} modes (95% of the training variance) "
        f"compresses {X.shape[1]}-pixel blood-cell images to "
        f"{100 * k_class / X.shape[1]:.0f}% of their size while a 1-NN classifier "
        f"in that basis reaches accuracy {acc_eigen:.2f} "
        f"(95% CI [{acc_lo:.2f}, {acc_hi:.2f}]) versus {acc_raw:.2f} on raw pixels, on the "
        "official held-out test partition."
    ),
    confidence="moderate",
    limitations_list=[
        f"Reaching 90/95/99% of variance needs {k90}/{k95}/{k99} modes "
        "respectively; the long scree tail means fine inter-class detail lives "
        "beyond the leading modes, so aggressive truncation costs the hardest cases.",
        "We averaged the colour channels to grayscale and used only a seeded "
        "few-hundred-image subsample; real blood-cell typing exploits colour and "
        "staining cues, and illumination and segmentation variation that a linear "
        "PCA basis does not model. Offline the analysis runs on the bundled "
        "fallback (see the provenance line above), not the real crops.",
        "1-NN is a deliberately simple recognizer chosen for transparency, not "
        "peak accuracy; it is sensitive to the distance metric and to class imbalance.",
        f"Accuracy is a single estimate on {y_test.size} images of the official test "
        f"partition (95% CI [{acc_lo:.2f}, {acc_hi:.2f}]); the interval is why this is "
        "reported at moderate, not high, confidence.",
        "The estimate bounds memorization, not every optimistic bias: BloodMNIST ships no "
        "donor or acquisition identifiers, so the same donor, near-duplicate crops, or shared "
        "staining and imaging conditions could still span the training and test partitions -- "
        "which is why we call it an honest held-out estimate, not 'leakage-free'.",
    ],
    evidence=(
        f"no preprocessing, PCA, mode selection, or classifier fitting used the official test "
        f"partition; basis, mean, and mode count fit on the training partition; "
        f"held-out 1-NN accuracy = {acc_eigen:.3f} at k={k_class} vs raw-pixel "
        f"baseline {acc_raw:.3f} (accuracy 95% CI [{acc_lo:.2f}, {acc_hi:.2f}], "
        f"n={y_test.size} test images); explained-variance milestones k90={k90}, "
        f"k95={k95}, k99={k99}."
    ),
)
print(block)

## Exercises

Your graded work for this week is **Problem Set 1 (PS1) -- "The Eigen-Subspace as a
Model of Normal Cells"**, distributed and auto-graded through GitHub Classroom. It keeps
this lesson's eigen-image basis but turns it to a new question: if the top principal axes
capture what a *normal* cell looks like, what does the **residual** -- the part that does not
fit -- tell you? The basis primitives (`eigen_basis`, `project`, `reconstruct`) and the data
loader are provided; you fill in the analysis on real BloodMNIST.

- **Part A -- denoise by low-rank projection.** A clean image lives in a few principal axes
  while additive noise spreads across all of them, so projecting a noisy crop onto the top-*k*
  subspace and reconstructing keeps the signal and discards most of the noise -- but only at
  the right rank. Implement `snr_db`, `denoise`, and `best_rank_for_denoising`, and explain
  why the SNR-versus-rank curve rises, peaks, and falls.
- **Part B -- flag out-of-QC images by reconstruction error.** An image that does not belong
  to the normal subspace reconstructs badly, so its reconstruction error is a novelty score
  for catching corrupted acquisitions (poor focus, saturation, sensor noise). Implement
  `reconstruction_anomaly_score`, `detection_auc` (the detector's ROC-AUC), and
  `flag_threshold` (a false-alarm-bounded cutoff), and close with an interpretation block at a
  defensible confidence level.

Refer to the PS1 repository README for the submission and auto-grading details.